# Day 4-01｜投籃影片上傳與格式轉換
> Python 籃球運動資料分析課程  
> 將指定的 `side.mp4` 整理成統一格式，供後續姿態與球軌跡分析使用。  
> 修課背景：具備基礎 Python 語法即可；不預設電腦視覺或運動資料分析經驗。

## 學習目標
- 把 `C:\Users\henry\Documents\side.mp4` 複製到 `assets/raw/`。
- 用 ffmpeg 轉成固定規格的 MP4。
- 確認 Day 4 / Day 5 會讀到同一支影片。

## 完成產出
- `assets/converted/video_001.mp4` 標準化影片檔。

## 課堂要求
- 按照本單元順序執行各段程式。
- 僅修改題目指定的變數、路徑或參數。
- 完成指定輸出後，記錄結果並供課堂討論。


## 執行階段提醒
請優先使用 **GPU** 或 **TPU** 的執行階段；不要使用純 CPU 執行。  
YOLO、MediaPipe 與影片處理在純 CPU 上會明顯較慢，容易讓課堂操作卡住。


## 課程流程
1. 檢查 `C:\Users\henry\Documents\side.mp4` 是否存在。
2. 複製到 `assets/raw/` 並轉成 `assets/converted/video_001.mp4`。
3. 確認 converted 影片清單。


In [1]:
from pathlib import Path
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/henry753951/basketball-hackathon-course.git"
REPO_BRANCH = "main"
IN_COLAB = False
DRIVE_MOUNTED = False

try:
    from google.colab import drive  # type: ignore[import-not-found]

    IN_COLAB = True
except ModuleNotFoundError:
    drive = None

if IN_COLAB and drive is not None:
    try:
        drive.mount("/content/drive")
        DRIVE_MOUNTED = True
    except NotImplementedError:
        print("目前這個 Colab runtime 不支援 google.colab.drive.mount，改用 /content 本機路徑。")

bootstrap_candidates = [
    Path.cwd().resolve(),
    *Path.cwd().resolve().parents,
    Path("/content/basketball_hackathon/course"),
    Path("/content/basketball_hackathon_course"),
    Path("/content/drive/MyDrive/basketball_hackathon/course"),
]

COURSE_ROOT_HINT = None
for candidate in bootstrap_candidates:
    if (candidate / "src" / "course_setup.py").exists():
        COURSE_ROOT_HINT = candidate
        break

if COURSE_ROOT_HINT is None and IN_COLAB:
    if DRIVE_MOUNTED:
        COURSE_ROOT_HINT = Path("/content/drive/MyDrive/basketball_hackathon/course")
    else:
        COURSE_ROOT_HINT = Path("/content/basketball_hackathon/course")

    if not (COURSE_ROOT_HINT / "src" / "course_setup.py").exists():
        if COURSE_ROOT_HINT.exists():
            shutil.rmtree(COURSE_ROOT_HINT)
        COURSE_ROOT_HINT.parent.mkdir(parents=True, exist_ok=True)
        cmd = [
            "git",
            "clone",
            "--depth",
            "1",
            "-b",
            REPO_BRANCH,
            REPO_URL,
            str(COURSE_ROOT_HINT),
        ]
        print("$", " ".join(cmd))
        subprocess.run(cmd, check=True)

if COURSE_ROOT_HINT is None or not (COURSE_ROOT_HINT / "src" / "course_setup.py").exists():
    raise FileNotFoundError(
        "找不到 src/course_setup.py。請先執行 init_colab.ipynb，或確認課程 repo 已經存在於目前目錄、/content/basketball_hackathon/course 或 Google Drive。"
    )

if str(COURSE_ROOT_HINT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT_HINT))

from src.course_setup import bootstrap_course_notebook  # noqa: E402

COURSE_ROOT = bootstrap_course_notebook(COURSE_ROOT_HINT, mount_drive=True)


課程根目錄: H:\Repos\basketball-hackathon-course
素材資料夾: H:\Repos\basketball-hackathon-course\assets
工具模組: H:\Repos\basketball-hackathon-course\src


## 預設示範影片

本課程預設直接使用這支影片：

```text
C:\Users\henry\Documents\side.mp4
```

執行下方程式後，會自動複製到：

```text
assets/raw/side.mp4
```

再轉成：

```text
assets/converted/video_001.mp4
```


In [2]:
from pathlib import Path
import shutil

from src.video_utils import convert_video, list_videos

RAW_DIR = COURSE_ROOT / "assets" / "raw"
CONVERTED_DIR = COURSE_ROOT / "assets" / "converted"
RAW_DIR.mkdir(parents=True, exist_ok=True)
CONVERTED_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_VIDEO = Path(r"C:\Users\henry\Documents\side.mp4")
RAW_VIDEO = RAW_DIR / SOURCE_VIDEO.name
OUTPUT_VIDEO = CONVERTED_DIR / "video_001.mp4"

if not SOURCE_VIDEO.exists():
    raise FileNotFoundError(
        f"找不到示範影片：{SOURCE_VIDEO}。請先確認檔案存在，再重新執行 Day 4-01。"
    )

if SOURCE_VIDEO.resolve() != RAW_VIDEO.resolve():
    shutil.copy2(SOURCE_VIDEO, RAW_VIDEO)
    print("copied:", RAW_VIDEO)
else:
    print("source video is already inside assets/raw:", RAW_VIDEO)

print("raw videos:")
for p in list_videos(RAW_DIR):
    print("-", p.name)


copied:

 H:\Repos\basketball-hackathon-course\assets\raw\side.mp4
raw videos:
- side.mp4


In [3]:
# 需要系統有 ffmpeg。Colab 通常已經有。
converted = convert_video(RAW_VIDEO, OUTPUT_VIDEO, fps=30, max_side=1280)
print("converted video:", converted)


$ C:\Users\henry\scoop\apps\ffmpeg-shared\8.1\bin\ffmpeg.exe -y -i H:\Repos\basketball-hackathon-course\assets\raw\side.mp4 -vf scale='if(gt(iw,ih),1280,-2)':'if(gt(ih,iw),1280,-2)',fps=30,format=yuv420p -an -vcodec libx264 -preset veryfast -crf 23 H:\Repos\basketball-hackathon-course\assets\converted\video_001.mp4


converted video: H:\Repos\basketball-hackathon-course\assets\converted\video_001.mp4


In [4]:
print("目前 converted 影片：")
for p in list_videos(CONVERTED_DIR):
    print("-", p.name)

目前 converted 影片：
- video_001.mp4


## 本單元產出檔案

- `assets/raw/side.mp4`：學生上傳後複製進 repo 的原始影片。
- `assets/converted/video_001.mp4`：統一格式、可直接後續分析的 converted 影片。
